In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="05-harmful-content-detection",
    notebook_name="03_multi_task_training_and_evaluation.ipynb"
)

# Multi-Task Training & Evaluation for Harmful Content Detection

## Quick Recap

In the previous two notebooks, we designed a **multi-task classifier** for harmful content detection. The model uses **early fusion** to combine text, image, and author features, then passes them through:

- **Shared layers** -- learn patterns useful for ALL harm categories (violence, nudity, hate speech)
- **Task-specific heads** -- one per harm category, each specializing in detecting that specific type of harm

### The School Analogy

Imagine a school with **one principal** (the shared layers) and **three specialist teachers** (the task heads).

The principal meets every student first and learns their general background -- are they energetic? quiet? artistic? This general understanding is useful for everyone. Then the student visits each specialist teacher:

- The **gym teacher** (violence head) checks if the student is being too aggressive
- The **art teacher** (nudity head) evaluates whether their artwork is appropriate
- The **language teacher** (hate speech head) listens to whether they're saying hurtful things

Each teacher uses the principal's general assessment PLUS their own expertise to make a decision. That's multi-task learning!

```
                  [Violence Head]    [Nudity Head]    [Hate Speech Head]
                   (gym teacher)     (art teacher)    (language teacher)
                       |                  |                  |
                  ===================================================
                  |           Shared Layers (principal)              |
                  ===================================================
                                     |
                              [Fused Features]
                              /       |       \
                        [Image]    [Text]   [Author]
```

In this notebook we will:
1. **Build** the multi-task model in PyTorch
2. **Train** it with BCE and Focal Loss
3. **Evaluate** with PR-AUC, ROC-AUC, and online metrics
4. **Simulate** the full serving pipeline
5. **Wrap up** with an interview cheat sheet

In [ ]:
###############################################################################
# Multi-Task Harmful Content Detector -- PyTorch Implementation
###############################################################################

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
torch.manual_seed(42)

HARM_CATEGORIES = ["violence", "nudity", "hate_speech"]


class HarmfulContentDetector(nn.Module):
    """
    Multi-task classifier for harmful content detection.

    Architecture
    ------------
    Shared layers   : Linear -> BatchNorm -> ReLU -> Dropout  (x2)
    Task-specific   : One small head per harm category
                      Linear -> ReLU -> Linear -> Sigmoid
    """

    def __init__(self, input_dim: int = 256, shared_dim: int = 128,
                 head_dim: int = 64, dropout: float = 0.3):
        super().__init__()

        # ---- shared layers (the "principal") ----
        self.shared = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(shared_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ---- task-specific heads (the "specialist teachers") ----
        self.heads = nn.ModuleDict()
        for task in HARM_CATEGORIES:
            self.heads[task] = nn.Sequential(
                nn.Linear(shared_dim, head_dim),
                nn.ReLU(),
                nn.Linear(head_dim, 1),
                nn.Sigmoid(),
            )

    def forward(self, x: torch.Tensor) -> dict:
        """Return dict {task_name: probability} for each harm category."""
        shared_out = self.shared(x)
        return {task: head(shared_out).squeeze(-1)
                for task, head in self.heads.items()}


# ---- instantiate & inspect ----
model = HarmfulContentDetector(input_dim=256)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Shared params:    {sum(p.numel() for p in model.shared.parameters()):,}")
for task in HARM_CATEGORIES:
    n = sum(p.numel() for p in model.heads[task].parameters())
    print(f"  {task:12s} head params: {n:,}")

## Dataset Construction

### Where do the labels come from?

Think of it like grading homework. There are two ways to do it:

| Strategy | How it works | Like... | Pros | Cons |
|----------|-------------|---------|------|------|
| **Natural labeling** | Use user reports and automated signals | Students grading each other | Free, scales to billions of posts | Noisy -- some reports are wrong, some harmful posts go unreported |
| **Hand labeling** | Paid human annotators review each post | The teacher grading it herself | Most accurate | Expensive and slow -- can't do it for every post |
| **Hybrid (our choice)** | Natural labels for training, hand labels for evaluation | Students help grade daily homework, teacher grades the final exam | Best of both worlds | Requires careful pipeline design |

**Why hybrid?**
- We have billions of posts -- we can't hand-label them all.
- User reports give us "free" labels at scale (even if noisy).
- But we need high-quality hand labels to *evaluate* whether the model is actually working.

### Class Imbalance

Here's the tricky part: **harmful posts are rare**. Imagine sorting through 1,000 marbles and only 10-50 of them are red (harmful). That's a 1-5% positive rate. This makes training harder because the model can just predict "not harmful" every time and still be 95% accurate! We'll address this with **focal loss** later.

In [ ]:
###############################################################################
# Synthetic Dataset -- mimicking the real class imbalance
###############################################################################

N_SAMPLES = 10_000
INPUT_DIM = 256

# Positive rates per task (harmful content is rare!)
POSITIVE_RATES = {"violence": 0.03, "nudity": 0.05, "hate_speech": 0.02}


def make_synthetic_dataset(n: int = N_SAMPLES, input_dim: int = INPUT_DIM,
                           positive_rates: dict = POSITIVE_RATES):
    """Create a synthetic multi-label dataset with realistic class imbalance."""
    # Features: random embeddings (stand-in for fused text+image+author features)
    X = np.random.randn(n, input_dim).astype(np.float32)

    # Labels: each task has its own positive rate
    labels = {}
    for task, rate in positive_rates.items():
        labels[task] = np.random.binomial(1, rate, size=n).astype(np.float32)

    # Make features slightly correlated with labels so the model can learn
    for task, y in labels.items():
        signal = np.random.randn(input_dim) * 0.5           # task-specific signal direction
        X[y == 1] += signal                                  # shift positives

    df = pd.DataFrame(labels)
    df.insert(0, "post_id", range(n))
    return torch.tensor(X), {t: torch.tensor(v) for t, v in labels.items()}, df


X_all, y_all, label_df = make_synthetic_dataset()

# ---- show class distribution ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, task in zip(axes, HARM_CATEGORIES):
    counts = label_df[task].value_counts().sort_index()
    colors = ["#3498db", "#e74c3c"]
    bars = ax.bar(["Safe (0)", "Harmful (1)"], counts.values, color=colors,
                  edgecolor="white", linewidth=1.2)
    ax.set_title(f"{task.replace('_', ' ').title()}\n"
                 f"({counts.values[1] / len(label_df) * 100:.1f}% positive)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                f"{int(bar.get_height()):,}", ha="center", fontsize=11)
fig.suptitle("Label Distribution -- harmful posts are rare!",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print(label_df.describe().round(3))

In [ ]:
###############################################################################
# Multi-Task Training Loop with BCE Loss
###############################################################################

def train_model(model, X, y_dict, loss_fn_class=nn.BCELoss, epochs=20,
                lr=1e-3, verbose=True):
    """
    Train the multi-task model.

    Multi-task loss = sum of per-task losses:
        L_total = L_violence + L_nudity + L_hate_speech
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # One loss function per task
    loss_fns = {task: loss_fn_class() for task in HARM_CATEGORIES}

    history = {task: [] for task in HARM_CATEGORIES}
    history["total"] = []

    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        preds = model(X)

        total_loss = torch.tensor(0.0)
        for task in HARM_CATEGORIES:
            task_loss = loss_fns[task](preds[task], y_dict[task])
            history[task].append(task_loss.item())
            total_loss = total_loss + task_loss

        history["total"].append(total_loss.item())
        total_loss.backward()
        optimizer.step()

        if verbose and (epoch % 5 == 0 or epoch == 1):
            parts = " | ".join(f"{t}: {history[t][-1]:.4f}" for t in HARM_CATEGORIES)
            print(f"Epoch {epoch:3d}  total={total_loss.item():.4f}  [{parts}]")

    return history


# --- Train with BCE ---
model_bce = HarmfulContentDetector(input_dim=INPUT_DIM)
print("=" * 70)
print("Training with standard Binary Cross-Entropy Loss")
print("=" * 70)
history_bce = train_model(model_bce, X_all, y_all, epochs=20)

# --- Plot per-task loss ---
fig, ax = plt.subplots(figsize=(10, 5))
for task in HARM_CATEGORIES:
    ax.plot(history_bce[task], label=task.replace("_", " ").title(), linewidth=2)
ax.plot(history_bce["total"], label="Total", linewidth=2.5, linestyle="--", color="black")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Per-Task Loss During Training (BCE)", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
###############################################################################
# Focal Loss -- Putting Extra Attention on the Hard (Rare) Examples
###############################################################################


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).

    Think of it like a teacher who spends MORE time helping students who
    struggle and LESS time on students who already ace the test.

    Standard BCE treats every example equally.
    Focal Loss down-weights easy (well-classified) examples and
    up-weights hard (misclassified) ones.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma = 0  -->  same as standard BCE
    gamma = 2  -->  common default, strongly focuses on hard examples
    alpha      -->  balances positive vs negative class weight
    """

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Clamp for numerical stability
        pred = pred.clamp(1e-7, 1 - 1e-7)

        # Binary cross-entropy (element-wise)
        bce = -target * torch.log(pred) - (1 - target) * torch.log(1 - pred)

        # p_t = probability of the TRUE class
        p_t = target * pred + (1 - target) * (1 - pred)

        # Focal modulating factor: (1 - p_t)^gamma
        # When the model is confident and correct, p_t ~ 1, so weight ~ 0 (ignore)
        # When the model is wrong, p_t ~ 0, so weight ~ 1 (focus here!)
        focal_weight = (1 - p_t) ** self.gamma

        # Alpha balancing for class imbalance
        alpha_t = target * self.alpha + (1 - target) * (1 - self.alpha)

        loss = alpha_t * focal_weight * bce
        return loss.mean()


# ---- Train with Focal Loss ----
model_focal = HarmfulContentDetector(input_dim=INPUT_DIM)
print("=" * 70)
print("Training with Focal Loss (gamma=2, alpha=0.25)")
print("=" * 70)
history_focal = train_model(model_focal, X_all, y_all,
                            loss_fn_class=FocalLoss, epochs=20)

# ---- Compare BCE vs Focal Loss ----
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, task in zip(axes, HARM_CATEGORIES):
    ax.plot(history_bce[task], label="BCE", linewidth=2)
    ax.plot(history_focal[task], label="Focal", linewidth=2, linestyle="--")
    ax.set_title(task.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("BCE vs Focal Loss per Task", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print("\nFocal loss focuses the model's attention on the HARD examples --")
print("the rare harmful posts that the model struggles to classify correctly.")
print("This is especially helpful when positive rates are 1-5%.")

In [ ]:
###############################################################################
# Gradient Blending -- Handling Modality Imbalance
###############################################################################

print("""
============================================================
 GRADIENT BLENDING -- Why We Need It
============================================================

Imagine three students working on a group project:
  - Student A (image features) learns super fast
  - Student B (text features)  learns at medium speed
  - Student C (author features) learns slowly

If Student A finishes early and the group stops studying,
Students B and C never learn properly!

In multimodal models, different modalities (image, text,
metadata) converge at different rates. The fast-learning
modality can DOMINATE the shared gradients, preventing the
slower modalities from contributing useful information.

Gradient blending fixes this by SCALING the gradients from
each modality so they contribute more evenly.
""")


# ---- Simple gradient scaling demonstration ----
# In practice, this is done inside the model's backward pass.
# Here we demonstrate the concept with toy gradients.

def simulate_gradient_blending(n_epochs=15):
    """
    Demonstrate how gradient blending balances learning rates
    across modalities.
    """
    modalities = ["image", "text", "author"]
    # Simulated gradient magnitudes over time (without blending)
    # Image features produce large gradients, author features produce small ones
    raw_grad_mag = {
        "image":  np.exp(-np.linspace(0, 2, n_epochs)) * 5.0,   # starts big, drops fast
        "text":   np.exp(-np.linspace(0, 1, n_epochs)) * 2.0,   # medium
        "author": np.exp(-np.linspace(0, 0.5, n_epochs)) * 0.5, # starts small, stays small
    }

    # Blending weights: inversely proportional to gradient magnitude
    # This makes large gradients smaller and small gradients larger
    blended_grad_mag = {}
    for m in modalities:
        # Scale each modality so they have similar magnitude
        scale = 1.0 / (raw_grad_mag[m].mean() + 1e-8)
        blended_grad_mag[m] = raw_grad_mag[m] * scale

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["#e74c3c", "#3498db", "#2ecc71"]

    for i, m in enumerate(modalities):
        axes[0].plot(raw_grad_mag[m], label=m.title(), color=colors[i], linewidth=2.5)
        axes[1].plot(blended_grad_mag[m], label=m.title(), color=colors[i], linewidth=2.5)

    axes[0].set_title("WITHOUT Gradient Blending\n(image dominates!)", fontweight="bold")
    axes[1].set_title("WITH Gradient Blending\n(balanced contributions)", fontweight="bold")
    for ax in axes:
        ax.set_xlabel("Epoch"); ax.set_ylabel("Gradient Magnitude")
        ax.legend(); ax.grid(alpha=0.3)

    fig.suptitle("Gradient Blending Balances Modality Learning Rates",
                 fontsize=14, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()


simulate_gradient_blending()

---

## Evaluation: How Do We Know the Model Is Working?

### Offline Metrics (measured BEFORE deployment)

We use two key metrics, each giving a different "camera angle" on the same model:

| Metric | What It Plots | Best For | Think of It As... |
|--------|---------------|----------|-----------|
| **PR-AUC** (Precision-Recall AUC) | Precision vs. Recall at every threshold | Imbalanced data (ours!) | "Of the posts I flagged, how many were actually harmful? And of the harmful ones, how many did I catch?" |
| **ROC-AUC** (Receiver Operating Characteristic AUC) | True Positive Rate vs. False Positive Rate | Overall discrimination ability | "Can the model tell harmful from safe? And does it avoid falsely accusing safe posts?" |

**Why use BOTH?**
- ROC-AUC can look misleadingly good on imbalanced data (because there are so many true negatives).
- PR-AUC focuses on the rare positive class, which is what we actually care about.
- Together, they give a complete picture.

**Analogy:** Imagine you're a hall monitor trying to catch students sneaking snacks into class.
- **PR-AUC** answers: "When you point at someone and say 'you have snacks!', how often are you right? And how many snack-smugglers did you miss?"
- **ROC-AUC** answers: "Can you reliably tell snack-smugglers from innocent students?"

In [ ]:
###############################################################################
# PR-AUC and ROC-AUC -- Visualized Per Task
###############################################################################

# Generate predictions from the focal-loss model
model_focal.eval()
with torch.no_grad():
    preds = model_focal(X_all)

# Convert to numpy
preds_np = {task: preds[task].numpy() for task in HARM_CATEGORIES}
labels_np = {task: y_all[task].numpy() for task in HARM_CATEGORIES}

# ---- PR Curves ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, task in zip(axes, HARM_CATEGORIES):
    precision, recall, _ = precision_recall_curve(labels_np[task], preds_np[task])
    ap = average_precision_score(labels_np[task], preds_np[task])

    ax.plot(recall, precision, linewidth=2.5, color="#e74c3c")
    ax.fill_between(recall, precision, alpha=0.15, color="#e74c3c")
    ax.set_title(f"{task.replace('_', ' ').title()}\nPR-AUC = {ap:.3f}",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("Recall (How many harmful posts did we catch?)")
    ax.set_ylabel("Precision (Of those we flagged, how many were really harmful?)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    # Baseline: random classifier PR-AUC = positive rate
    baseline = labels_np[task].mean()
    ax.axhline(y=baseline, linestyle="--", color="gray", alpha=0.5,
               label=f"Random baseline ({baseline:.3f})")
    ax.legend(loc="upper right")

fig.suptitle("Precision-Recall Curves (Higher = Better)",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# ---- ROC Curves ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, task in zip(axes, HARM_CATEGORIES):
    fpr, tpr, _ = roc_curve(labels_np[task], preds_np[task])
    auc_val = roc_auc_score(labels_np[task], preds_np[task])

    ax.plot(fpr, tpr, linewidth=2.5, color="#3498db")
    ax.fill_between(fpr, tpr, alpha=0.15, color="#3498db")
    ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5, label="Random (0.500)")
    ax.set_title(f"{task.replace('_', ' ').title()}\nROC-AUC = {auc_val:.3f}",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("False Positive Rate (wrongly flagged safe posts)")
    ax.set_ylabel("True Positive Rate (correctly caught harmful posts)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

fig.suptitle("ROC Curves (Hugs the Top-Left = Better)",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# ---- Summary table ----
print("\n" + "=" * 50)
print(f"{'Task':15s} {'PR-AUC':>10s} {'ROC-AUC':>10s}")
print("=" * 50)
for task in HARM_CATEGORIES:
    pr = average_precision_score(labels_np[task], preds_np[task])
    roc = roc_auc_score(labels_np[task], preds_np[task])
    print(f"{task:15s} {pr:10.3f} {roc:10.3f}")
print("=" * 50)

In [ ]:
###############################################################################
# Online Metrics -- What We Track After Deployment
###############################################################################


def compute_online_metrics(total_posts, total_impressions,
                           harmful_detected, harmful_missed,
                           harmful_impressions_prevented,
                           total_harmful_impressions,
                           appeals_reversed, total_appeals,
                           system_found, user_reported):
    """
    Compute the online metrics that matter in production.

    These are what the execs and policy teams look at every week.
    """
    prevalence = harmful_missed / total_posts
    harmful_imp = (total_harmful_impressions - harmful_impressions_prevented
                   ) / total_impressions
    valid_appeal_rate = appeals_reversed / max(harmful_detected, 1)
    proactive_rate = system_found / max(system_found + user_reported, 1)

    return {
        "prevalence": prevalence,
        "harmful_impressions": harmful_imp,
        "valid_appeal_rate": valid_appeal_rate,
        "proactive_rate": proactive_rate,
    }


# ---- Simulation: watch metrics improve as the model gets better ----
print("Simulating model improvements over 6 quarters...\n")

# Each quarter the model catches more harmful content and makes fewer mistakes
quarters = ["Q1-2025", "Q2-2025", "Q3-2025", "Q4-2025", "Q1-2026", "Q2-2026"]
simulation_data = []

np.random.seed(42)
for i, q in enumerate(quarters):
    # Model improves each quarter
    detection_quality = 0.5 + i * 0.08  # gets better over time

    total_posts = 500_000_000
    total_impressions = 50_000_000_000
    true_harmful = int(total_posts * 0.03)   # 3% of posts are truly harmful

    harmful_detected = int(true_harmful * detection_quality)
    harmful_missed = true_harmful - harmful_detected
    total_harmful_impressions = int(true_harmful * 1000)  # avg 1000 impressions per harmful post
    harmful_impressions_prevented = int(total_harmful_impressions * detection_quality)
    false_positives = int(harmful_detected * (0.15 - i * 0.015))  # decreasing FP rate
    system_found = int(harmful_detected * (0.6 + i * 0.05))
    user_reported = harmful_detected - system_found

    metrics = compute_online_metrics(
        total_posts, total_impressions,
        harmful_detected, harmful_missed,
        harmful_impressions_prevented, total_harmful_impressions,
        false_positives, harmful_detected,
        system_found, user_reported
    )
    metrics["quarter"] = q
    simulation_data.append(metrics)

sim_df = pd.DataFrame(simulation_data).set_index("quarter")

# ---- Visualize ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_info = [
    ("prevalence", "Prevalence\n(lower is better)", "#e74c3c",
     "Fraction of harmful posts that slip through"),
    ("harmful_impressions", "Harmful Impressions\n(lower is better)", "#e67e22",
     "Fraction of total impressions that are harmful"),
    ("valid_appeal_rate", "Valid Appeal Rate\n(lower is better)", "#9b59b6",
     "How often we wrongly remove posts"),
    ("proactive_rate", "Proactive Rate\n(higher is better)", "#2ecc71",
     "Fraction caught by system (not user reports)"),
]

for ax, (col, title, color, desc) in zip(axes.flat, metric_info):
    vals = sim_df[col].values
    ax.bar(range(len(quarters)), vals, color=color, alpha=0.8, edgecolor="white")
    ax.set_xticks(range(len(quarters)))
    ax.set_xticklabels(quarters, rotation=30)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_ylabel(col.replace("_", " ").title())
    ax.grid(axis="y", alpha=0.3)
    # Add annotation
    ax.text(0.5, 0.95, desc, transform=ax.transAxes, fontsize=9,
            ha="center", va="top", style="italic", color="gray")

fig.suptitle("Online Metrics Over Time -- The Model Gets Better Each Quarter",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(sim_df.round(4).to_string())

---

## Serving Architecture

Once the model is trained and evaluated, it needs to run in production -- processing every single post in near real-time. Here's how the system works:

```
User creates post (text, image, video)
        |
        v
+---------------------------------------+
| Harmful Content Detection Service     |
|  - Extract features (text, image,     |
|    author, context)                   |
|  - Run multi-task ML model            |
|  - Get harm probabilities per task    |
+---------------------------------------+
        |
        +---> p(harm) > HIGH threshold (e.g., 0.95)
        |         |
        |         v
        |    +-----------------------------+
        |    | Violation Enforcement       |
        |    |  - IMMEDIATELY remove post  |
        |    |  - Notify user (explain     |
        |    |    which rule was broken)   |
        |    |  - Log for appeals          |
        |    +-----------------------------+
        |
        +---> LOW threshold < p(harm) < HIGH threshold
        |         |
        |         v
        |    +-----------------------------+
        |    | Demoting Service            |
        |    |  - Reduce post visibility   |
        |    |  - Queue for human review   |
        |    |  - Post still accessible    |
        |    |    but shown to fewer users |
        |    +-----------------------------+
        |
        +---> p(harm) < LOW threshold (e.g., 0.3)
                  |
                  v
             Post goes live normally
```

### The Three Components

| Component | When It Activates | What It Does | Analogy |
|-----------|-------------------|-------------|----------|
| **Detection Service** | Every new post | Predicts harm probability | The metal detector at the entrance |
| **Violation Enforcement** | High confidence harmful | Removes post, notifies user | The bouncer who throws you out |
| **Demoting Service** | Low confidence harmful | Reduces visibility, queues for review | The teacher who says "sit in the corner until I check your homework" |

In [ ]:
###############################################################################
# End-to-End Pipeline Simulation
###############################################################################


class HarmfulContentPipeline:
    """
    Simulates the full production pipeline:
    Post -> Detection -> Enforce / Demote / Allow
    """

    def __init__(self, model, high_threshold=0.8, low_threshold=0.3):
        self.model = model
        self.high_threshold = high_threshold
        self.low_threshold = low_threshold

    def process_batch(self, features: torch.Tensor) -> pd.DataFrame:
        """Process a batch of posts and return decisions."""
        self.model.eval()
        with torch.no_grad():
            preds = self.model(features)

        results = []
        n = features.shape[0]
        for i in range(n):
            # Get max harm probability across all tasks
            task_probs = {task: preds[task][i].item() for task in HARM_CATEGORIES}
            max_task = max(task_probs, key=task_probs.get)
            max_prob = task_probs[max_task]

            # Decision logic
            if max_prob >= self.high_threshold:
                decision = "REMOVE"
                action = f"Violation enforcement: {max_task}"
            elif max_prob >= self.low_threshold:
                decision = "DEMOTE"
                action = f"Queue for review: {max_task}"
            else:
                decision = "ALLOW"
                action = "Post goes live"

            results.append({
                "post_id": i,
                "max_harm_prob": round(max_prob, 4),
                "top_category": max_task,
                "decision": decision,
                "action": action,
                **{f"p_{t}": round(task_probs[t], 4) for t in HARM_CATEGORIES},
            })

        return pd.DataFrame(results)


# ---- Run the pipeline on a batch of posts ----
pipeline = HarmfulContentPipeline(model_focal, high_threshold=0.8, low_threshold=0.3)

# Use a subset for clarity
batch_size = 200
batch_X = X_all[:batch_size]
results_df = pipeline.process_batch(batch_X)

# ---- Show decision distribution ----
print("Pipeline Decision Distribution:")
print(results_df["decision"].value_counts().to_string())
print()

# Show some example decisions
print("\nSample Decisions (first 10 posts):")
print(results_df[["post_id", "max_harm_prob", "top_category", "decision", "action"]]
      .head(10).to_string(index=False))

# ---- Visualize decision distribution ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision pie chart
decision_counts = results_df["decision"].value_counts()
colors_pie = {"ALLOW": "#2ecc71", "DEMOTE": "#f39c12", "REMOVE": "#e74c3c"}
axes[0].pie(decision_counts.values,
            labels=decision_counts.index,
            colors=[colors_pie.get(d, "gray") for d in decision_counts.index],
            autopct="%1.1f%%", startangle=90,
            textprops={"fontsize": 12, "fontweight": "bold"})
axes[0].set_title("Pipeline Decisions", fontweight="bold", fontsize=13)

# Probability histogram
axes[1].hist(results_df["max_harm_prob"], bins=30, color="#3498db",
             edgecolor="white", alpha=0.8)
axes[1].axvline(x=0.3, color="#f39c12", linestyle="--", linewidth=2,
                label=f"Low threshold ({0.3})")
axes[1].axvline(x=0.8, color="#e74c3c", linestyle="--", linewidth=2,
                label=f"High threshold ({0.8})")
axes[1].set_xlabel("Max Harm Probability")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Harm Scores", fontweight="bold", fontsize=13)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ---- Confusion matrix (using the "any harmful" aggregation) ----
# Ground truth: is the post harmful for ANY task?
batch_labels = np.stack([y_all[t][:batch_size].numpy() for t in HARM_CATEGORIES], axis=1)
true_harmful = (batch_labels.sum(axis=1) > 0).astype(int)
pred_harmful = (results_df["decision"] != "ALLOW").astype(int).values

cm = confusion_matrix(true_harmful, pred_harmful)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Safe", "Harmful"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix\n(Any Harmful Category)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nTrue harmful posts in batch: {true_harmful.sum()} / {batch_size}")
print(f"Posts flagged (REMOVE or DEMOTE): {pred_harmful.sum()} / {batch_size}")

---

## Interview Cheat Sheet

### The 7-Step Framework (memorize this!)

```
1. CLARIFY    What modalities? What harm types? Latency needs? Languages?
2. FRAME      Multi-task classification with early fusion of modalities
3. DATA       Users + Posts + Interactions tables
              Natural labels (user reports) for training
              Hand labels (human annotators) for evaluation
4. FEATURES   Text embeddings (DistilBERT) + Image embeddings (CLIP/SimCLR)
              + User reactions + Author features + Contextual features
5. MODEL      Neural network: shared layers + task-specific heads
6. TRAIN      BCE or Focal Loss per task, sum of losses
              Handle modality imbalance with gradient blending
7. EVALUATE   Offline: PR-AUC + ROC-AUC per task
              Online:  Prevalence, Harmful impressions, Valid appeals, Proactive rate
8. SERVE      Detection service -> High confidence: Violation Enforcement (remove)
                                -> Low confidence:  Demoting Service (reduce visibility)
```

### Common Follow-Up Questions

| Question | Strong Answer |
|----------|---------------|
| **Why early fusion over late fusion?** | Memes and combined content can be harmful even when each modality is benign in isolation. Early fusion captures cross-modal interactions. Late fusion misses this entirely. |
| **Why multi-task over separate models?** | Shared layers enable knowledge transfer between tasks. Cheaper to maintain (one model, not N). Limited labeled data for one task benefits from data in other tasks. |
| **How do you handle class imbalance?** | Focal loss (down-weight easy examples, up-weight hard ones). Oversampling. Strategic threshold tuning per harm category. |
| **How do you handle adversarial content?** | Obfuscation detection (l33t speak, Unicode tricks). Character normalization. Image perturbation augmentation during training. |
| **How do you decide the confidence threshold?** | Tune per category based on cost of false positives vs. false negatives. Violence gets a LOW threshold (better safe than sorry). Satire/controversy gets a HIGHER threshold (don't over-censor). |
| **What about appeals?** | Track valid appeal rate as an online metric. High appeal rate = model is too aggressive (too many false positives). Use appeal outcomes to retrain. |
| **How do you handle new types of harmful content?** | Active learning pipeline: surface uncertain predictions for human review. Retrain periodically with newly labeled data. Add new task heads without retraining shared layers. |
| **What about latency?** | Feature extraction at post creation time. Model inference is fast (single forward pass through the network). Heavy content (video) processed asynchronously. |

### Key Phrases to Drop in an Interview

- "We use **early fusion** because cross-modal context matters -- a benign image plus benign text can be a harmful meme."
- "The **multi-task architecture** with shared layers enables knowledge transfer and is more cost-effective than separate models."
- "We train with **focal loss** to handle the extreme class imbalance where harmful posts are only 1-5% of all content."
- "We evaluate with **both PR-AUC and ROC-AUC** -- PR-AUC is more informative for imbalanced data."
- "Our online metrics focus on **harmful impressions** (not just prevalence) because a viral harmful post is much worse than one seen by 10 people."
- "We use a **two-tier enforcement** system: high-confidence violations are removed immediately; low-confidence ones are demoted and queued for human review."
- "**Gradient blending** prevents the fast-learning modality from dominating training."
- "The **valid appeal rate** is our key false-positive signal -- if users successfully appeal too often, the model is too aggressive."

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)